# UFC Fight Dataset – Comprehensive EDA

This notebook performs an in‑depth exploratory data analysis (EDA) of the `ufc-master.csv` dataset, 
with a focus on:

- Understanding the schema and column groups (fighters, betting odds, rankings, stats, etc.)
- Visualizing key distributions (weight class, finish types, odds, etc.)
- Exploring domain‑specific MMA patterns:
  - Reach / height / age advantages
  - Stance matchups
  - Title bouts vs non‑title bouts
  - Fight location & country effects
  - Striking and grappling stats vs outcomes

The goal is to learn patterns that will later help design AI/ML models to **predict the fight winner**.

## 0. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Optional: uncomment if you have seaborn installed and want nicer plots
# import seaborn as sns
# sns.set(style="whitegrid")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Adjust path as needed when you run this locally
csv_path = "ufc-master.csv"  # put the CSV in the same folder as this notebook
df = pd.read_csv(csv_path)

df.head()

## 1. Basic Dataset Structure

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe(include=['number'])

In [ ]:
df.describe(include=['object', 'bool'])

## 2. Missing Values

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

In [ ]:
# Visualize missingness for top columns with missing values
top_missing = missing[missing > 0].head(25).index
df[top_missing].isna().mean().plot(kind="bar", figsize=(10,4))
plt.title("Fraction of Missing Values (Top 25 Columns)")
plt.ylabel("Fraction missing")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 3. Column Groups & High‑Level Description

In [ ]:
# Quick programmatic grouping of columns by prefix
cols = df.columns

groups = {
    "metadata": [c for c in cols if c in [
        "Date", "Location", "Country", "Winner", "TitleBout", "WeightClass",
        "Gender", "NumberOfRounds", "Finish", "FinishDetails",
        "FinishRound", "FinishRoundTime", "TotalFightTimeSecs", "EmptyArena"
    ]],
    "fighters_core": [c for c in cols if c in ["RedFighter", "BlueFighter"]],
    "odds": [c for c in cols if "Odds" in c or "ExpectedValue" in c],
    "blue_stats": [c for c in cols if c.startswith("Blue") and c not in ["BlueFighter"]],
    "red_stats": [c for c in cols if c.startswith("Red") and c not in ["RedFighter"]],
    "diff_features": [c for c in cols if c.endswith("Dif")],
    "ranks": [c for c in cols if "Rank" in c],
}

groups

In [ ]:
for name, cols_in_group in groups.items():
    print(f"\n=== {name.upper()} ({len(cols_in_group)} columns) ===")
    print(cols_in_group)

## 4. Core Categorical Distributions

In [ ]:
df['Winner'].value_counts(normalize=False)

In [ ]:
(df['Winner'].value_counts(normalize=True) * 100).round(2)

In [ ]:
df['Winner'].value_counts().plot(kind="bar", figsize=(4,3))
plt.title("Winner Distribution (Blue vs Red)")
plt.ylabel("Number of Fights")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
df['WeightClass'].value_counts().head(20)

In [ ]:
df['WeightClass'].value_counts().plot(kind="bar", figsize=(10,4))
plt.title("Weight Class Distribution")
plt.ylabel("Number of Fights")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
df['TitleBout'].value_counts()

In [ ]:
df['TitleBout'].value_counts().plot(kind="bar", figsize=(4,3))
plt.title("Title Bout vs Non‑Title")
plt.ylabel("Number of Fights")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
df['Gender'].value_counts()

In [ ]:
df['NumberOfRounds'].value_counts().sort_index()

In [ ]:
df['NumberOfRounds'].value_counts().sort_index().plot(kind="bar", figsize=(5,3))
plt.title("Scheduled Number of Rounds")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
df['Country'].value_counts().head(15)

In [ ]:
df['Country'].value_counts().head(15).plot(kind="bar", figsize=(8,3))
plt.title("Top 15 Fight Host Countries")
plt.ylabel("Number of Fights")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
df['Finish'].value_counts()

In [ ]:
df['Finish'].value_counts().plot(kind="bar", figsize=(6,3))
plt.title("Finish Type Distribution")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 5. Betting Odds & Market Expectations

In [ ]:
df[['RedOdds', 'BlueOdds']].describe()

In [ ]:
df['RedOdds'].hist(bins=40, figsize=(6,4), alpha=0.7)
plt.title("Red Odds Distribution")
plt.xlabel("American Odds (Red)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

df['BlueOdds'].hist(bins=40, figsize=(6,4), alpha=0.7)
plt.title("Blue Odds Distribution")
plt.xlabel("American Odds (Blue)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Convert American odds to implied probability (simplified, ignoring overround)
def american_to_prob(odds):
    if pd.isna(odds):
        return np.nan
    if odds < 0:
        return (-odds) / ((-odds) + 100)
    else:
        return 100 / (odds + 100)

df['RedImpliedProb'] = df['RedOdds'].apply(american_to_prob)
df['BlueImpliedProb'] = df['BlueOdds'].apply(american_to_prob)

df[['RedImpliedProb', 'BlueImpliedProb']].describe()

In [ ]:
# Check calibration: how often does Blue win when the market favors Blue?
df['BlueFavouredByOdds'] = df['BlueImpliedProb'] > df['RedImpliedProb']
df['BlueWin'] = (df['Winner'] == 'Blue').astype(int)

df.groupby('BlueFavouredByOdds')['BlueWin'].agg(['count', 'mean'])

## 6. Striking & Grappling Statistics

In [ ]:
strike_cols = [
    'BlueAvgSigStrLanded', 'BlueAvgSigStrPct',
    'RedAvgSigStrLanded', 'RedAvgSigStrPct',
    'SigStrDif'
]
df[strike_cols].describe()

In [ ]:
df['BlueAvgSigStrLanded'].hist(bins=30, figsize=(5,3), alpha=0.7)
plt.title("BlueAvgSigStrLanded Distribution")
plt.xlabel("Sig. Strikes Landed per Min (?)")
plt.tight_layout()
plt.show()

df['RedAvgSigStrLanded'].hist(bins=30, figsize=(5,3), alpha=0.7)
plt.title("RedAvgSigStrLanded Distribution")
plt.xlabel("Sig. Strikes Landed per Min (?)")
plt.tight_layout()
plt.show()

In [ ]:
df['SigStrDif'].hist(bins=40, figsize=(6,3))
plt.title("SigStrDif Distribution (Blue - Red)")
plt.xlabel("Blue - Red Sig. Strikes (Avg)")
plt.tight_layout()
plt.show()

In [ ]:
grap_cols = [
    'BlueAvgSubAtt', 'BlueAvgTDLanded', 'BlueAvgTDPct',
    'RedAvgSubAtt', 'RedAvgTDLanded', 'RedAvgTDPct',
    'SubDif', 'AvgSubAttDif', 'AvgTDDif'
]
df[grap_cols].describe()

In [ ]:
df['BlueAvgTDLanded'].hist(bins=30, figsize=(5,3), alpha=0.7)
plt.title("BlueAvgTDLanded Distribution")
plt.xlabel("Avg TD Landed")
plt.tight_layout()
plt.show()

df['RedAvgTDLanded'].hist(bins=30, figsize=(5,3), alpha=0.7)
plt.title("RedAvgTDLanded Distribution")
plt.xlabel("Avg TD Landed")
plt.tight_layout()
plt.show()

In [ ]:
df['AvgTDDif'].hist(bins=40, figsize=(6,3))
plt.title("AvgTDDif Distribution (Blue - Red)")
plt.xlabel("Blue - Red Avg TD Landed")
plt.tight_layout()
plt.show()

In [ ]:
df['BlueWinsBySubmission'].describe(), df['RedWinsBySubmission'].describe()

## 7. Physical Attributes: Height, Reach, Weight, Age

In [ ]:
phys_cols = [
    'RedHeightCms', 'BlueHeightCms',
    'RedReachCms', 'BlueReachCms',
    'RedWeightLbs', 'BlueWeightLbs',
    'RedAge', 'BlueAge',
    'HeightDif', 'ReachDif', 'AgeDif'
]
df[phys_cols].describe()

In [ ]:
df['RedAge'].hist(bins=30, figsize=(5,3), alpha=0.7)
plt.title("RedAge Distribution")
plt.xlabel("Age (years)")
plt.tight_layout()
plt.show()

df['BlueAge'].hist(bins=30, figsize=(5,3), alpha=0.7)
plt.title("BlueAge Distribution")
plt.xlabel("Age (years)")
plt.tight_layout()
plt.show()

In [ ]:
df['AgeDif'].hist(bins=40, figsize=(6,3))
plt.title("AgeDif Distribution (Blue - Red)")
plt.xlabel("Age Difference (Blue - Red)")
plt.tight_layout()
plt.show()

In [ ]:
df['HeightDif'].hist(bins=40, figsize=(6,3))
plt.title("HeightDif Distribution (Blue - Red)")
plt.xlabel("Height Difference (cm, Blue - Red)")
plt.tight_layout()
plt.show()

df['ReachDif'].hist(bins=40, figsize=(6,3))
plt.title("ReachDif Distribution (Blue - Red)")
plt.xlabel("Reach Difference (cm, Blue - Red)")
plt.tight_layout()
plt.show()

## 8. Domain-Specific Advantage Analysis (Age, Height, Reach vs Winner)

In [ ]:
# Ensure BlueWin exists
df['BlueWin'] = (df['Winner'] == 'Blue').astype(int)

def summarize_advantage(diff_col, bins, bin_label):
    tmp = df.copy()
    tmp[bin_label] = pd.cut(tmp[diff_col], bins=bins, include_lowest=True)
    summary = tmp.groupby(bin_label)['BlueWin'].agg(['count', 'mean'])
    summary = summary.rename(columns={'mean': 'BlueWinRate'})
    return summary

age_bins = [-50, -10, -5, 0, 5, 10, 50]
height_bins = [-50, -10, -5, 0, 5, 10, 50]
reach_bins = [-50, -10, -5, 0, 5, 10, 50]

age_summary = summarize_advantage('AgeDif', age_bins, 'AgeDiffBin')
height_summary = summarize_advantage('HeightDif', height_bins, 'HeightDiffBin')
reach_summary = summarize_advantage('ReachDif', reach_bins, 'ReachDiffBin')

age_summary, height_summary, reach_summary

In [ ]:
reach_summary['BlueWinRate'].plot(kind="bar", figsize=(6,3))
plt.title("Blue Win Rate by Reach Difference Bin (Blue - Red)")
plt.ylabel("Blue Win Rate")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
height_summary['BlueWinRate'].plot(kind="bar", figsize=(6,3))
plt.title("Blue Win Rate by Height Difference Bin (Blue - Red)")
plt.ylabel("Blue Win Rate")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
age_summary['BlueWinRate'].plot(kind="bar", figsize=(6,3))
plt.title("Blue Win Rate by Age Difference Bin (Blue - Red)")
plt.ylabel("Blue Win Rate")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 9. Stance Matchups

In [ ]:
df['RedStance'].value_counts()

In [ ]:
df['BlueStance'].value_counts()

In [ ]:
# Combined stance matchups
df['StanceMatchup'] = df['RedStance'].fillna('Unknown') + "_vs_" + df['BlueStance'].fillna('Unknown')

stance_summary = (
    df.groupby('StanceMatchup')['BlueWin']
      .agg(['count', 'mean'])
      .rename(columns={'mean': 'BlueWinRate'})
      .sort_values('count', ascending=False)
)

stance_summary.head(20)

In [ ]:
stance_summary.head(10)['count'].plot(kind="bar", figsize=(8,3))
plt.title("Most Common Stance Matchups (Count)")
plt.ylabel("Number of Fights")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 10. Title Fights vs Non‑Title Fights

In [ ]:
title_summary = (
    df.groupby('TitleBout')['BlueWin']
      .agg(['count', 'mean'])
      .rename(columns={'mean': 'BlueWinRate'})
)
title_summary

In [ ]:
# Finish types broken down by title bout
finish_title = pd.crosstab(df['Finish'], df['TitleBout'], normalize='columns') * 100
finish_title.round(2)

In [ ]:
finish_title.plot(kind="bar", figsize=(8,4))
plt.title("Finish Type (%) – Title vs Non‑Title")
plt.ylabel("Percentage within Column")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 11. Location & Country Effects

In [ ]:
country_summary = (
    df.groupby('Country')['BlueWin']
      .agg(['count', 'mean'])
      .rename(columns={'mean': 'BlueWinRate'})
      .sort_values('count', ascending=False)
)

country_summary.head(20)

In [ ]:
country_summary.head(15)['count'].plot(kind="bar", figsize=(8,3))
plt.title("Top 15 Countries by Number of Fights")
plt.ylabel("Number of Fights")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
country_summary.head(15)['BlueWinRate'].plot(kind="bar", figsize=(8,3))
plt.title("Blue Win Rate in Top 15 Countries")
plt.ylabel("Blue Win Rate")
plt.ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 12. Finish Type vs Weight Class & Scheduled Rounds

In [ ]:
finish_wc = pd.crosstab(df['WeightClass'], df['Finish'], normalize='index') * 100
finish_wc.round(1).head(15)

In [ ]:
finish_wc.head(10).plot(kind="bar", stacked=True, figsize=(10,5))
plt.title("Finish Type by Weight Class (%)")
plt.ylabel("Percentage within Weight Class")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
finish_rounds = pd.crosstab(df['NumberOfRounds'], df['Finish'], normalize='index') * 100
finish_rounds.round(1)

In [ ]:
finish_rounds.plot(kind="bar", stacked=True, figsize=(7,4))
plt.title("Finish Type by Scheduled Number of Rounds (%)")
plt.ylabel("Percentage within Rounds Category")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 13. Correlation Heatmap of Key Features

In [ ]:
key_numeric_cols = [
    'BlueImpliedProb', 'RedImpliedProb',
    'AgeDif', 'HeightDif', 'ReachDif',
    'SigStrDif', 'AvgSubAttDif', 'AvgTDDif',
    'LoseStreakDif', 'WinStreakDif',
    'KODif', 'SubDif',
    'TotalFightTimeSecs',
    'NumberOfRounds',
    'BlueWin'
]

corr_df = df[key_numeric_cols].copy()
corr = corr_df.corr()

plt.figure(figsize=(10,8))
plt.imshow(corr, interpolation='nearest')
plt.xticks(range(len(key_numeric_cols)), key_numeric_cols, rotation=90)
plt.yticks(range(len(key_numeric_cols)), key_numeric_cols)
plt.title("Correlation Heatmap – Selected Features")
plt.colorbar()
plt.tight_layout()
plt.show()

corr

## 14. Notes & Ideas for Winner Prediction Modeling

Based on this EDA, we can note down some potentially useful ideas for AI/ML modeling:

- **Symmetric representation:**  
  The dataset already contains many *difference* features (e.g., `AgeDif`, `ReachDif`, `SigStrDif`, `AvgTDDif`).  
  These are ideal for models that predict whether **Blue** wins (`BlueWin` as target), because they encode
  "advantage" for Blue vs Red.

- **Betting odds as a strong baseline:**  
  Columns like `BlueImpliedProb` and `RedImpliedProb` often correlate strongly with actual outcomes.  
  A simple model that uses only these as inputs can serve as a baseline to beat.

- **Physical advantages:**  
  Binned analyses of `ReachDif`, `HeightDif`, and `AgeDif` reveal how much physical advantages matter in practice.  
  These can be incorporated directly as features or via non‑linear transformations / interaction terms.

- **Style (stance) and skill profile:**  
  "StanceMatchup" and stats like `SigStrDif`, `AvgTDDif`, `AvgSubAttDif` capture different styles (striker, wrestler, grappler).  
  Models may benefit from explicitly engineered style features.

- **Contextual factors:**  
  Features like `TitleBout`, `WeightClass`, `Country`, and `NumberOfRounds` can be encoded via one‑hot encoding
  or target encoding to capture how context changes the likelihood of a Blue win.

These insights will guide the **feature engineering and model design** in the next phase.